# Condition 2 Analysis (2026-03-21)

Largest condition 2 sample to date: 26 subjects, all condition 2.

Design (same as Mar 13 pilot):
1. **Double study pass**: all 60 study pairs presented twice (120 study trials total)
2. **Balanced test split**: 30 intact + 30 rearranged (10/10 per emotion)

**Prior results:** Mar 11 (n=6, single study, 24/36 bug) and Mar 13 (n=6, double study, 30/30 fix) both showed d' at zero. Combined n=12 with zero associative discrimination. This sample (n=26) provides substantially more power.

### Analyses

**Study phase (orienting).** 2(flanker gender compatibility) x 3(flanker emotion) RM ANOVAs on accuracy and RT.

**Test phase (associative recognition).** 2(pair type: intact/rearranged) x 3(flanker emotion) RM ANOVAs on p("same") and RT.

**Supplementary.** Signal detection analysis (d', criterion) per emotion.

In [1]:
import pandas as pd
from pathlib import Path
from statistics import NormalDist

z = NormalDist().inv_cdf

# Resolve notebook directory robustly
if '__vsc_ipynb_file__' in dir():
    _nb_dir = Path(__vsc_ipynb_file__).parent
else:
    # When run via nbclient/jupyter execute, look for the CSV in data/
    _candidates = [Path.cwd(), Path.cwd() / 'data', Path(__file__).parent if '__file__' in dir() else Path.cwd()]
    _nb_dir = next((p for p in _candidates if (p / '2026_03_21_26subj_c2.csv').exists()), Path.cwd())

df = pd.read_csv(_nb_dir / '2026_03_21_26subj_c2.csv')
df['correct'] = df['correct'].astype('boolean')

print(f'{len(df)} rows, {df.subject_number.nunique()} subjects')
print(f'Conditions: {df.groupby("condition").subject_number.nunique().to_dict()}')
print()

# Verify study-phase double presentation
study_counts = df[df.phase == 'study'].groupby('subject_number').size()
print(f'Study trials per subject: {study_counts.unique()} (expected 120 = 60 pairs x 2 passes)')
print()

# Verify test-phase split
test_all = df[df.phase == 'test']
pt_counts = test_all.groupby(['subject_number', 'pair_type']).size().unstack(fill_value=0)
print('Test-phase intact/rearranged per subject:')
print(pt_counts.to_string())
print()
emo_pt = test_all.groupby(['flanker_emotion', 'pair_type']).size().unstack(fill_value=0)
print('Test-phase counts by emotion x pair_type (all subjects):')
print(emo_pt.to_string())

4680 rows, 26 subjects
Conditions: {2: 26}

Study trials per subject: [120] (expected 120 = 60 pairs x 2 passes)

Test-phase intact/rearranged per subject:
pair_type       intact  rearranged
subject_number                    
1                   30          30
2                   30          30
3                   30          30
4                   30          30
5                   30          30
6                   30          30
7                   30          30
8                   30          30
9                   30          30
10                  30          30
11                  30          30
12                  30          30
13                  30          30
14                  30          30
15                  30          30
16                  30          30
17                  30          30
18                  30          30
19                  30          30
20                  30          30
21                  30          30
22                  30          30
23  

## Exclusion Criteria

Same criterion as prior pilots: exclude subjects with >=6 zero-correct cells out of 12 in the study-phase design (2 target gender x 2 flanker gender x 3 flanker emotion). With the double study pass, each cell has 10 trials.

In [2]:
ZERO_CELL_THRESHOLD = 6

study_all = df[df.phase == 'study']

cell_correct = study_all.groupby(
    ['subject_number', 'target_gender', 'flanker_gender', 'flanker_emotion']
).correct.sum().reset_index()
zero_cells = cell_correct.groupby('subject_number').apply(
    lambda g: (g.correct == 0).sum()
).reset_index(name='zero_cells')

subj_study_acc = study_all.groupby('subject_number').correct.mean()
zero_cells['study_accuracy'] = zero_cells.subject_number.map(subj_study_acc)

trials_per_cell = study_all.groupby(
    ['subject_number', 'target_gender', 'flanker_gender', 'flanker_emotion']
).size().iloc[0]
print(f'Study-phase design: 12 cells per subject ({trials_per_cell} trials each)')
print(f'Subjects with zero-correct cells:')
has_zeros = zero_cells[zero_cells.zero_cells > 0].sort_values('zero_cells', ascending=False)
if len(has_zeros) == 0:
    print('  None')
else:
    for _, row in has_zeros.iterrows():
        flag = ' ** EXCLUDED' if row.zero_cells >= ZERO_CELL_THRESHOLD else ''
        print(f'  Subject {int(row.subject_number)}: '
              f'{int(row.zero_cells)}/12 zero cells, {row.study_accuracy:.1%} accuracy{flag}')
print()

excluded = zero_cells[zero_cells.zero_cells >= ZERO_CELL_THRESHOLD].subject_number.tolist()
keep = zero_cells[zero_cells.zero_cells < ZERO_CELL_THRESHOLD].subject_number.tolist()
df = df[df.subject_number.isin(keep)].copy()
print(f'{len(excluded)} excluded, {df.subject_number.nunique()} subjects remain')

Study-phase design: 12 cells per subject (10 trials each)
Subjects with zero-correct cells:
  Subject 25: 6/12 zero cells, 46.7% accuracy ** EXCLUDED
  Subject 7: 4/12 zero cells, 51.7% accuracy
  Subject 12: 4/12 zero cells, 52.5% accuracy
  Subject 26: 4/12 zero cells, 50.0% accuracy
  Subject 23: 2/12 zero cells, 50.8% accuracy
  Subject 24: 1/12 zero cells, 60.0% accuracy

1 excluded, 25 subjects remain


### Exclusion summary

1 subject excluded (Subject 25: 6/12 zero-correct cells, 46.7% study accuracy — chance-level responding). Five other subjects had 1-4 zero cells but did not meet the 6-cell threshold. 25 subjects remain for analysis.

## Study Phase (Orienting)

2(flanker gender compatibility) x 3(flanker emotion) RM ANOVAs on accuracy and RT. Each cell has 10 trials (double study pass).

In [3]:
study = df[df.phase == 'study'].copy()
study['compatible'] = study.target_gender == study.flanker_gender
study['compat_label'] = study.compatible.map({True: 'compatible', False: 'incompatible'})

study_acc_subj = study.groupby(
    ['subject_number', 'compat_label', 'flanker_emotion']
).correct.mean().reset_index(name='accuracy')

study_rt_subj = study[~study.timed_out].groupby(
    ['subject_number', 'compat_label', 'flanker_emotion']
).rt.mean().reset_index(name='mean_rt')

acc_table = study_acc_subj.groupby(['compat_label', 'flanker_emotion']).accuracy.agg(
    ['mean', 'std']
).round(3)
print('Study-phase accuracy by compatibility x emotion:')
print(acc_table.to_string())
print()

print('Marginal means (accuracy):')
print(f'  Compatible:   {study_acc_subj[study_acc_subj.compat_label == "compatible"].accuracy.mean():.3f}')
print(f'  Incompatible: {study_acc_subj[study_acc_subj.compat_label == "incompatible"].accuracy.mean():.3f}')
for emo in ['angry', 'happy', 'neutral']:
    print(f'  {emo:>12}: {study_acc_subj[study_acc_subj.flanker_emotion == emo].accuracy.mean():.3f}')
print()

rt_table = study_rt_subj.groupby(['compat_label', 'flanker_emotion']).mean_rt.agg(
    ['mean', 'std']
).round(1)
print('Study-phase RT (ms) by compatibility x emotion:')
print(rt_table.to_string())
print()

print('Marginal means (RT):')
print(f'  Compatible:   {study_rt_subj[study_rt_subj.compat_label == "compatible"].mean_rt.mean():.1f}')
print(f'  Incompatible: {study_rt_subj[study_rt_subj.compat_label == "incompatible"].mean_rt.mean():.1f}')
for emo in ['angry', 'happy', 'neutral']:
    print(f'  {emo:>12}: {study_rt_subj[study_rt_subj.flanker_emotion == emo].mean_rt.mean():.1f}')

Study-phase accuracy by compatibility x emotion:
                               mean    std
compat_label flanker_emotion              
compatible   angry            0.942  0.089
             happy             0.97  0.065
             neutral          0.946  0.073
incompatible angry            0.756  0.353
             happy            0.776  0.349
             neutral          0.764   0.36

Marginal means (accuracy):
  Compatible:   0.953
  Incompatible: 0.765
         angry: 0.849
         happy: 0.873
       neutral: 0.855

Study-phase RT (ms) by compatibility x emotion:
                                mean    std
compat_label flanker_emotion               
compatible   angry            1021.1  259.8
             happy             997.3  243.2
             neutral          1010.9  278.0
incompatible angry            1108.6  344.9
             happy            1103.9  351.5
             neutral          1124.5  350.6

Marginal means (RT):
  Compatible:   1009.8
  Incompatible: 1112.3


In [4]:
import math

def _betacf(a, b, x):
    MAXIT, EPS = 200, 3e-12
    qab, qap, qam = a + b, a + 1.0, a - 1.0
    c = 1.0
    d = 1.0 / (1.0 - qab * x / qap) if abs(1.0 - qab * x / qap) > 1e-30 else 1.0 / 1e-30
    h = d
    for m in range(1, MAXIT + 1):
        m2 = 2 * m
        aa = m * (b - m) * x / ((qam + m2) * (a + m2))
        d = 1.0 + aa * d
        if abs(d) < 1e-30: d = 1e-30
        c = 1.0 + aa / c
        if abs(c) < 1e-30: c = 1e-30
        d = 1.0 / d
        h *= d * c
        aa = -(a + m) * (qab + m) * x / ((a + m2) * (qap + m2))
        d = 1.0 + aa * d
        if abs(d) < 1e-30: d = 1e-30
        c = 1.0 + aa / c
        if abs(c) < 1e-30: c = 1e-30
        d = 1.0 / d
        delta = d * c
        h *= delta
        if abs(delta - 1.0) < EPS:
            break
    return h

def _betai(a, b, x):
    if x <= 0: return 0.0
    if x >= 1: return 1.0
    lbeta = math.lgamma(a) + math.lgamma(b) - math.lgamma(a + b)
    front = math.exp(a * math.log(x) + b * math.log(1 - x) - lbeta)
    if x < (a + 1) / (a + b + 2):
        return front * _betacf(a, b, x) / a
    else:
        return 1.0 - front * _betacf(b, a, 1 - x) / b

def t_p_twotail(t_val, df):
    x = df / (df + t_val ** 2)
    return _betai(df / 2.0, 0.5, x)

def f_p(f_val, df1, df2):
    if f_val <= 0: return 1.0
    x = df2 / (df2 + df1 * f_val)
    return _betai(df2 / 2.0, df1 / 2.0, x)

def rm_anova_oneway(groups):
    k = len(groups)
    n = len(groups[0])
    grand = sum(sum(g) for g in groups) / (k * n)
    subj_m = [sum(groups[j][i] for j in range(k)) / k for i in range(n)]
    cond_m = [sum(g) / n for g in groups]
    ss_cond = n * sum((m - grand) ** 2 for m in cond_m)
    ss_subj = k * sum((m - grand) ** 2 for m in subj_m)
    ss_total = sum((groups[j][i] - grand) ** 2 for j in range(k) for i in range(n))
    ss_err = ss_total - ss_cond - ss_subj
    df1 = k - 1
    df2 = (k - 1) * (n - 1)
    ms_err = ss_err / df2 if df2 > 0 else float('nan')
    f_val = (ss_cond / df1) / ms_err if ss_err > 0 else float('nan')
    p = f_p(f_val, df1, df2)
    eta = ss_cond / (ss_cond + ss_err)
    return f_val, df1, df2, p, eta, ms_err

def rm_anova_twoway(data, a_levels, b_levels):
    a = len(a_levels)
    b = len(b_levels)
    n = len(data[(a_levels[0], b_levels[0])])
    Y = [[[data[(a_levels[j], b_levels[k])][i]
           for k in range(b)] for j in range(a)] for i in range(n)]
    gm = sum(Y[i][j][k] for i in range(n) for j in range(a) for k in range(b)) / (n * a * b)
    subj_m = [sum(Y[i][j][k] for j in range(a) for k in range(b)) / (a * b) for i in range(n)]
    a_m = [sum(Y[i][j][k] for i in range(n) for k in range(b)) / (n * b) for j in range(a)]
    b_m = [sum(Y[i][j][k] for i in range(n) for j in range(a)) / (n * a) for k in range(b)]
    ab_m = [[sum(Y[i][j][k] for i in range(n)) / n for k in range(b)] for j in range(a)]
    sa_m = [[sum(Y[i][j][k] for k in range(b)) / b for j in range(a)] for i in range(n)]
    sb_m = [[sum(Y[i][j][k] for j in range(a)) / a for k in range(b)] for i in range(n)]
    ss_a = n * b * sum((a_m[j] - gm) ** 2 for j in range(a))
    ss_b = n * a * sum((b_m[k] - gm) ** 2 for k in range(b))
    ss_ab = n * sum((ab_m[j][k] - a_m[j] - b_m[k] + gm) ** 2
                    for j in range(a) for k in range(b))
    ss_s = a * b * sum((subj_m[i] - gm) ** 2 for i in range(n))
    ss_as = b * sum((sa_m[i][j] - a_m[j] - subj_m[i] + gm) ** 2
                    for i in range(n) for j in range(a))
    ss_bs = a * sum((sb_m[i][k] - b_m[k] - subj_m[i] + gm) ** 2
                    for i in range(n) for k in range(b))
    ss_total = sum((Y[i][j][k] - gm) ** 2
                   for i in range(n) for j in range(a) for k in range(b))
    ss_abs = ss_total - ss_a - ss_b - ss_ab - ss_s - ss_as - ss_bs
    df_a, df_b, df_ab = a - 1, b - 1, (a - 1) * (b - 1)
    df_as, df_bs, df_abs = df_a * (n - 1), df_b * (n - 1), df_ab * (n - 1)
    results = {}
    for label, ss_eff, df_eff, ss_e, df_e in [
        ('A', ss_a, df_a, ss_as, df_as),
        ('B', ss_b, df_b, ss_bs, df_bs),
        ('AxB', ss_ab, df_ab, ss_abs, df_abs),
    ]:
        ms_eff = ss_eff / df_eff if df_eff > 0 else 0
        ms_e = ss_e / df_e if df_e > 0 else float('nan')
        f_val = ms_eff / ms_e if ms_e > 0 else float('nan')
        p = f_p(f_val, df_eff, df_e)
        eta = ss_eff / (ss_eff + ss_e) if (ss_eff + ss_e) > 0 else 0
        results[label] = {
            'F': f_val, 'df1': df_eff, 'df2': df_e,
            'p': p, 'eta_sq': eta, 'ms_error': ms_e
        }
    return results

def anova_followup(means_a, means_b, ms_error, df_error, label_a, label_b):
    n = len(means_a)
    diff = sum(a - b for a, b in zip(means_a, means_b)) / n
    se = math.sqrt(2 * ms_error / n)
    if se == 0:
        return 0.0, df_error, 1.0, diff
    t_val = diff / se
    p = t_p_twotail(t_val, df_error)
    return t_val, df_error, p, diff

def edge_correct(rate, n):
    if rate == 0:
        return 0.5 / n
    if rate == 1:
        return 1 - 0.5 / n
    return rate


# --- Study-phase 2x3 ANOVAs ---

subjects = sorted(study_acc_subj.subject_number.unique())
n_subj = len(subjects)
a_levels = ['compatible', 'incompatible']
b_levels = ['angry', 'happy', 'neutral']

acc_data = {}
for al in a_levels:
    for bl in b_levels:
        mask = (study_acc_subj.compat_label == al) & (study_acc_subj.flanker_emotion == bl)
        vals = study_acc_subj[mask].set_index('subject_number').loc[subjects, 'accuracy'].tolist()
        acc_data[(al, bl)] = vals

print(f'2(compatibility) x 3(emotion) RM ANOVA on accuracy (n={n_subj}):')
print()
acc_results = rm_anova_twoway(acc_data, a_levels, b_levels)
for label, name in [('A', 'Compatibility'), ('B', 'Emotion'), ('AxB', 'Compatibility x Emotion')]:
    r = acc_results[label]
    print(f"  {name}: F({r['df1']},{r['df2']}) = {r['F']:.3f}, "
          f"p = {r['p']:.3f}, partial eta^2 = {r['eta_sq']:.3f}")
print()

rt_data = {}
for al in a_levels:
    for bl in b_levels:
        mask = (study_rt_subj.compat_label == al) & (study_rt_subj.flanker_emotion == bl)
        vals = study_rt_subj[mask].set_index('subject_number').loc[subjects, 'mean_rt'].tolist()
        rt_data[(al, bl)] = vals

print(f'2(compatibility) x 3(emotion) RM ANOVA on RT (n={n_subj}):')
print()
rt_results = rm_anova_twoway(rt_data, a_levels, b_levels)
for label, name in [('A', 'Compatibility'), ('B', 'Emotion'), ('AxB', 'Compatibility x Emotion')]:
    r = rt_results[label]
    print(f"  {name}: F({r['df1']},{r['df2']}) = {r['F']:.3f}, "
          f"p = {r['p']:.3f}, partial eta^2 = {r['eta_sq']:.3f}")

2(compatibility) x 3(emotion) RM ANOVA on accuracy (n=25):

  Compatibility: F(1,24) = 6.896, p = 0.015, partial eta^2 = 0.223
  Emotion: F(2,48) = 2.670, p = 0.080, partial eta^2 = 0.100
  Compatibility x Emotion: F(2,48) = 0.313, p = 0.733, partial eta^2 = 0.013

2(compatibility) x 3(emotion) RM ANOVA on RT (n=25):

  Compatibility: F(1,24) = 12.248, p = 0.002, partial eta^2 = 0.338
  Emotion: F(2,48) = 0.686, p = 0.508, partial eta^2 = 0.028
  Compatibility x Emotion: F(2,48) = 0.316, p = 0.730, partial eta^2 = 0.013


### Study phase interpretation

The flanker-compatibility effect is now significant with the larger sample. Compatible trials show high accuracy (.95) while incompatible trials are lower (.77), F(1,24) = 6.90, p = .015, partial eta^2 = .22. The RT compatibility effect is also significant: compatible faster (1010 ms) than incompatible (1112 ms), F(1,24) = 12.25, p = .002, partial eta^2 = .34. No main effect of emotion and no interaction in either measure.

These study-phase results replicate and strengthen the prior pilot findings. The orienting task is working as intended — subjects are engaged and the flanker-compatibility manipulation produces robust effects on both accuracy and RT.

## Test Phase (Associative Recognition)

Each test trial shows a face pair. Intact pairs are the same target-flanker combination seen at study; rearranged pairs swap the flanker identity within the same trial type. The participant judges "same" (intact) or "different" (rearranged).

2(pair type: intact/rearranged) x 3(flanker emotion) RM ANOVAs on p("same") and RT.

**Cell sizes.** 10 intact + 10 rearranged per emotion per subject (balanced).

In [5]:
test = df[df.phase == 'test'].copy()

test['said_same'] = test.apply(
    lambda r: bool(r.correct) if r.pair_type == 'intact' else not bool(r.correct), axis=1
)

psame_subj = test.groupby(
    ['subject_number', 'pair_type', 'flanker_emotion']
).said_same.mean().reset_index(name='p_same')

rt_test_subj = test[~test.timed_out].groupby(
    ['subject_number', 'pair_type', 'flanker_emotion']
).rt.mean().reset_index(name='mean_rt')

print('Test-phase p("same") by pair_type x emotion:')
psame_table = psame_subj.groupby(['pair_type', 'flanker_emotion']).p_same.agg(
    ['mean', 'std']
).round(3)
print(psame_table.to_string())
print()

print('Marginal means p("same"):')
print(f'  Intact:     {psame_subj[psame_subj.pair_type == "intact"].p_same.mean():.3f}')
print(f'  Rearranged: {psame_subj[psame_subj.pair_type == "rearranged"].p_same.mean():.3f}')
for emo in ['angry', 'happy', 'neutral']:
    print(f'  {emo:>12}: {psame_subj[psame_subj.flanker_emotion == emo].p_same.mean():.3f}')
print()

print('Test-phase RT (ms) by pair_type x emotion:')
rt_table = rt_test_subj.groupby(['pair_type', 'flanker_emotion']).mean_rt.agg(
    ['mean', 'std']
).round(1)
print(rt_table.to_string())
print()

print('Marginal means RT (ms):')
print(f'  Intact:     {rt_test_subj[rt_test_subj.pair_type == "intact"].mean_rt.mean():.1f}')
print(f'  Rearranged: {rt_test_subj[rt_test_subj.pair_type == "rearranged"].mean_rt.mean():.1f}')
for emo in ['angry', 'happy', 'neutral']:
    print(f'  {emo:>12}: {rt_test_subj[rt_test_subj.flanker_emotion == emo].mean_rt.mean():.1f}')

Test-phase p("same") by pair_type x emotion:
                             mean    std
pair_type  flanker_emotion              
intact     angry            0.548  0.228
           happy            0.512  0.219
           neutral          0.508  0.223
rearranged angry            0.496  0.223
           happy            0.484  0.251
           neutral          0.504  0.184

Marginal means p("same"):
  Intact:     0.523
  Rearranged: 0.495
         angry: 0.522
         happy: 0.498
       neutral: 0.506

Test-phase RT (ms) by pair_type x emotion:
                              mean    std
pair_type  flanker_emotion               
intact     angry            1419.8  269.1
           happy            1429.1  302.9
           neutral          1389.4  308.5
rearranged angry            1385.2  306.8
           happy            1421.4  325.3
           neutral          1373.6  255.4

Marginal means RT (ms):
  Intact:     1412.8
  Rearranged: 1393.4
         angry: 1402.5
         happy: 1425.2
 

In [6]:
test_subjects = sorted(psame_subj.subject_number.unique())
n_test = len(test_subjects)
pt_levels = ['intact', 'rearranged']
emo_levels = ['angry', 'happy', 'neutral']

psame_data = {}
for pt in pt_levels:
    for emo in emo_levels:
        mask = (psame_subj.pair_type == pt) & (psame_subj.flanker_emotion == emo)
        vals = psame_subj[mask].set_index('subject_number').loc[test_subjects, 'p_same'].tolist()
        psame_data[(pt, emo)] = vals

print(f'2(pair_type) x 3(emotion) RM ANOVA on p("same") (n={n_test}):')
print()
psame_results = rm_anova_twoway(psame_data, pt_levels, emo_levels)
for label, name in [('A', 'Pair type'), ('B', 'Emotion'), ('AxB', 'Pair type x Emotion')]:
    r = psame_results[label]
    print(f"  {name}: F({r['df1']},{r['df2']}) = {r['F']:.3f}, "
          f"p = {r['p']:.3f}, partial eta^2 = {r['eta_sq']:.3f}")
print()

rt_data_test = {}
for pt in pt_levels:
    for emo in emo_levels:
        mask = (rt_test_subj.pair_type == pt) & (rt_test_subj.flanker_emotion == emo)
        vals = rt_test_subj[mask].set_index('subject_number').loc[test_subjects, 'mean_rt'].tolist()
        rt_data_test[(pt, emo)] = vals

print(f'2(pair_type) x 3(emotion) RM ANOVA on RT (n={n_test}):')
print()
rt_test_results = rm_anova_twoway(rt_data_test, pt_levels, emo_levels)
for label, name in [('A', 'Pair type'), ('B', 'Emotion'), ('AxB', 'Pair type x Emotion')]:
    r = rt_test_results[label]
    print(f"  {name}: F({r['df1']},{r['df2']}) = {r['F']:.3f}, "
          f"p = {r['p']:.3f}, partial eta^2 = {r['eta_sq']:.3f}")
print()

mse_int_psame = psame_results['AxB']['ms_error']
df_int_psame = psame_results['AxB']['df2']
mse_int_rt = rt_test_results['AxB']['ms_error']
df_int_rt = rt_test_results['AxB']['df2']

print('Follow-up comparisons on p("same"):')
print()
print('  Intact vs rearranged within each emotion (discrimination):')
for emo in emo_levels:
    t, dfe, p, md = anova_followup(
        psame_data[('intact', emo)], psame_data[('rearranged', emo)],
        mse_int_psame, df_int_psame, f'intact-{emo}', f'rearranged-{emo}'
    )
    print(f'    {emo}: t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.3f}')
print()

print('  Pairwise emotion within intact (hit rate modulation):')
emo_pairs = [('angry', 'happy'), ('angry', 'neutral'), ('happy', 'neutral')]
for e1, e2 in emo_pairs:
    t, dfe, p, md = anova_followup(
        psame_data[('intact', e1)], psame_data[('intact', e2)],
        mse_int_psame, df_int_psame, e1, e2
    )
    print(f'    {e1} vs {e2}: t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.3f}')
print()

print('  Pairwise emotion within rearranged (FA rate modulation):')
for e1, e2 in emo_pairs:
    t, dfe, p, md = anova_followup(
        psame_data[('rearranged', e1)], psame_data[('rearranged', e2)],
        mse_int_psame, df_int_psame, e1, e2
    )
    print(f'    {e1} vs {e2}: t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.3f}')
print()

print('Follow-up comparisons on RT:')
print()
print('  Intact vs rearranged within each emotion:')
for emo in emo_levels:
    t, dfe, p, md = anova_followup(
        rt_data_test[('intact', emo)], rt_data_test[('rearranged', emo)],
        mse_int_rt, df_int_rt, f'intact-{emo}', f'rearranged-{emo}'
    )
    print(f'    {emo}: t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.1f} ms')
print()

print('  Pairwise emotion within intact:')
for e1, e2 in emo_pairs:
    t, dfe, p, md = anova_followup(
        rt_data_test[('intact', e1)], rt_data_test[('intact', e2)],
        mse_int_rt, df_int_rt, e1, e2
    )
    print(f'    {e1} vs {e2}: t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.1f} ms')
print()

print('  Pairwise emotion within rearranged:')
for e1, e2 in emo_pairs:
    t, dfe, p, md = anova_followup(
        rt_data_test[('rearranged', e1)], rt_data_test[('rearranged', e2)],
        mse_int_rt, df_int_rt, e1, e2
    )
    print(f'    {e1} vs {e2}: t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.1f} ms')

2(pair_type) x 3(emotion) RM ANOVA on p("same") (n=25):

  Pair type: F(1,24) = 1.278, p = 0.270, partial eta^2 = 0.051
  Emotion: F(2,48) = 0.544, p = 0.584, partial eta^2 = 0.022
  Pair type x Emotion: F(2,48) = 0.455, p = 0.637, partial eta^2 = 0.019

2(pair_type) x 3(emotion) RM ANOVA on RT (n=25):

  Pair type: F(1,24) = 0.987, p = 0.330, partial eta^2 = 0.039
  Emotion: F(2,48) = 1.694, p = 0.195, partial eta^2 = 0.066
  Pair type x Emotion: F(2,48) = 0.115, p = 0.891, partial eta^2 = 0.005

Follow-up comparisons on p("same"):

  Intact vs rearranged within each emotion (discrimination):
    angry: t(48) = 1.462, p = 0.150, diff = 0.052
    happy: t(48) = 0.787, p = 0.435, diff = 0.028
    neutral: t(48) = 0.112, p = 0.911, diff = 0.004

  Pairwise emotion within intact (hit rate modulation):
    angry vs happy: t(48) = 1.012, p = 0.317, diff = 0.036
    angry vs neutral: t(48) = 1.125, p = 0.266, diff = 0.040
    happy vs neutral: t(48) = 0.112, p = 0.911, diff = 0.004

  Pairwi

### Test phase interpretation

**Associative discrimination remains near zero with n=25.** The pair type main effect on p("same") is F(1,24) = 1.28, p = .270 — subjects responded "same" to 52.3% of intact pairs and 49.5% of rearranged pairs. The 2.8 percentage point difference is not significant and the effect size is small (eta^2 = .051). No emotion effects and no interaction.

The angry condition shows the largest (but non-significant) intact-rearranged difference: 5.2 pp, t(48) = 1.46, p = .150. Happy and neutral are essentially at zero.

RT shows the same null pattern: no pair type effect (F < 1, p = .330), no interaction (F < 1, p = .891). Intact (1413 ms) and rearranged (1393 ms) RTs are not significantly different.

**Comparison to prior pilots.** The Mar 11 (n=6) and Mar 13 (n=6) pilots showed p("same") differences of exactly 0% and 0% respectively. The current sample shows a slight 2.8% difference that is not significant. With n=25, this study had adequate power to detect moderate effects — the continued null strongly suggests there is no associative memory signal in this paradigm.

## Supplementary: Signal Detection Analysis

d' and criterion per emotion. Hit = p("same" | intact), FA = p("same" | rearranged). Edge correction: Macmillan & Kaplan (1985). Cell sizes: 10 intact + 10 rearranged per emotion per subject.

In [7]:
sdt_per_subj = []
for subj in test_subjects:
    sdata = test[test.subject_number == subj]
    for emotion in emo_levels:
        intact_emo = sdata[(sdata.pair_type == 'intact') & (sdata.flanker_emotion == emotion)]
        rearr_emo = sdata[(sdata.pair_type == 'rearranged') & (sdata.flanker_emotion == emotion)]

        n_intact = len(intact_emo)
        n_rearr = len(rearr_emo)

        hit_rate_raw = intact_emo.said_same.mean() if n_intact > 0 else 0.0
        fa_rate_raw = rearr_emo.said_same.mean() if n_rearr > 0 else 0.0

        hit_rate = edge_correct(hit_rate_raw, n_intact) if n_intact > 0 else 0.5
        fa_rate = edge_correct(fa_rate_raw, n_rearr) if n_rearr > 0 else 0.5

        dprime = z(hit_rate) - z(fa_rate)
        criterion = -0.5 * (z(hit_rate) + z(fa_rate))

        sdt_per_subj.append({
            'subject': subj,
            'emotion': emotion,
            'n_intact': n_intact,
            'n_rearranged': n_rearr,
            'hit_rate': hit_rate_raw,
            'fa_rate': fa_rate_raw,
            'd_prime': dprime,
            'criterion': criterion
        })

sdt_df = pd.DataFrame(sdt_per_subj)

sdt_summary = sdt_df.groupby('emotion').agg(
    N=('subject', 'count'),
    hit_rate_M=('hit_rate', 'mean'),
    hit_rate_SD=('hit_rate', 'std'),
    fa_rate_M=('fa_rate', 'mean'),
    fa_rate_SD=('fa_rate', 'std'),
    d_prime_M=('d_prime', 'mean'),
    d_prime_SD=('d_prime', 'std'),
    criterion_M=('criterion', 'mean'),
    criterion_SD=('criterion', 'std'),
).round(3)

print(f'SDT Analysis (n={n_test} subjects, per-emotion hit and FA rates)')
print(f'Cell sizes: {sdt_df.n_intact.iloc[0]} intact, {sdt_df.n_rearranged.iloc[0]} rearranged per emotion per subject')
print()
print(sdt_summary.to_string())
print()

dprime_wide = sdt_df.pivot(index='subject', columns='emotion', values='d_prime')
angry_d = dprime_wide['angry'].tolist()
happy_d = dprime_wide['happy'].tolist()
neutral_d = dprime_wide['neutral'].tolist()

f_val_d, df1_d, df2_d, p_d, eta_d, mse_d = rm_anova_oneway([angry_d, happy_d, neutral_d])
print(f"One-way RM ANOVA on d' (flanker emotion):")
print(f"  F({df1_d},{df2_d}) = {f_val_d:.3f}, p = {p_d:.3f}, partial eta^2 = {eta_d:.3f}")
print()

print("Follow-up comparisons on d' (using omnibus MSE):")
d_groups = [angry_d, happy_d, neutral_d]
d_labels = ['angry', 'happy', 'neutral']
for i in range(3):
    for j in range(i + 1, 3):
        t, dfe, p, md = anova_followup(d_groups[i], d_groups[j], mse_d, df2_d,
                                        d_labels[i], d_labels[j])
        print(f'  {d_labels[i]} vs {d_labels[j]}: t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.3f}')
print()

crit_wide = sdt_df.pivot(index='subject', columns='emotion', values='criterion')
angry_c = crit_wide['angry'].tolist()
happy_c = crit_wide['happy'].tolist()
neutral_c = crit_wide['neutral'].tolist()

f_val_c, df1_c, df2_c, p_c, eta_c, mse_c = rm_anova_oneway([angry_c, happy_c, neutral_c])
print(f"One-way RM ANOVA on criterion (flanker emotion):")
print(f"  F({df1_c},{df2_c}) = {f_val_c:.3f}, p = {p_c:.3f}, partial eta^2 = {eta_c:.3f}")

SDT Analysis (n=25 subjects, per-emotion hit and FA rates)
Cell sizes: 10 intact, 10 rearranged per emotion per subject

          N  hit_rate_M  hit_rate_SD  fa_rate_M  fa_rate_SD  d_prime_M  d_prime_SD  criterion_M  criterion_SD
emotion                                                                                                      
angry    25       0.548        0.228      0.496       0.223      0.139       0.547       -0.072         0.627
happy    25       0.512        0.219      0.484       0.251      0.108       0.660       -0.007         0.629
neutral  25       0.508        0.223      0.504       0.184     -0.004       0.448       -0.022         0.574

One-way RM ANOVA on d' (flanker emotion):
  F(2,48) = 0.525, p = 0.595, partial eta^2 = 0.021

Follow-up comparisons on d' (using omnibus MSE):
  angry vs happy: t(48) = 0.211, p = 0.834, diff = 0.031
  angry vs neutral: t(48) = 0.974, p = 0.335, diff = 0.143
  happy vs neutral: t(48) = 0.763, p = 0.449, diff = 0.112

One-way 

### SDT interpretation

d' values are near zero for all three emotion conditions: angry d' = 0.14, happy d' = 0.11, neutral d' = -0.00. The one-way ANOVA on d' shows no emotion effect, F(2,48) = 0.53, p = .595. Criterion is near zero for all emotions (no response bias), F(2,48) = 0.51, p = .602.

There is a slight positive drift in angry and happy d' compared to the prior pilots (which were slightly negative), but none approach significance. Hit rates (.51-.55) and FA rates (.48-.50) remain close to .50, consistent with near-chance performance.

**Comparison to prior pilots (combined n=12):**
- Mar 11+13 d': angry ≈ -0.05, happy ≈ -0.08, neutral ≈ 0.01
- Mar 21 d': angry = 0.14, happy = 0.11, neutral = -0.00

The slight positive shift in angry/happy d' is not significant and is within the range expected from sampling variability. Across all three data collections (total n=37 before exclusions), the consistent finding is that d' hovers around zero with no reliable discrimination.

## Summary

### Practical implications

With n=25 analyzed subjects (the largest condition 2 sample to date), associative discrimination remains non-significant. This confirms and extends the prior pilot findings (combined n=12, d' ≈ 0).

**The power argument is now much weaker.** With n=25, even a small-to-medium effect (d' ≈ 0.3) should be detectable. The observed d' values (0.14, 0.11, -0.00) are small and non-significant, with large individual variability (SDs 0.45-0.66). The pair type main effect on p("same") has an effect size of eta^2 = .051, which is small.

**The study phase works.** Compatibility effects on both accuracy (p = .015) and RT (p = .002) are now significant with the larger sample, confirming that the flanker manipulation is effective during encoding.

**The dissociation with condition 1 is robust.** Condition 1 (n=32) showed reliable item recognition (d' ≈ 0.44, p < .001). Condition 2 (now n=25) shows no associative recognition. Subjects encode individual faces but do not bind them into retrievable pair representations under incidental encoding with the gender-judgment flanker task.

**Across all condition 2 data (n=37 collected, ~35 analyzed after exclusions):** The consistent null strongly suggests that this paradigm does not produce measurable associative memory. The failure is not due to insufficient power, insufficient study exposure, or the 24/36 split bug — it is a fundamental limitation of the incidental encoding task for associative binding.

In [8]:
n_total = 26
n_analyzed = df.subject_number.nunique()
n_excluded = n_total - n_analyzed

print(f'=== Condition 2 Summary (2026-03-21) ===')
print(f'{n_total} subjects collected (all c2), {n_excluded} excluded, {n_analyzed} analyzed')
print(f'Design: double study pass (120 trials), balanced 30/30 test split')
print()

print('Study phase (orienting):')
print(f'  Overall accuracy: {study.correct.mean():.1%}')
print(f'  Mean RT: {study.loc[~study.timed_out, "rt"].mean():.0f} ms')
r = acc_results['A']
print(f"  Compatibility: F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}")
r = acc_results['B']
print(f"  Emotion:       F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}")
r = acc_results['AxB']
print(f"  Interaction:   F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}")
print()

print('Test phase (associative recognition):')
r = psame_results['A']
print(f"  p(\"same\") Pair type:       F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}, eta^2 = {r['eta_sq']:.3f}")
r = psame_results['B']
print(f"  p(\"same\") Emotion:          F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}, eta^2 = {r['eta_sq']:.3f}")
r = psame_results['AxB']
print(f"  p(\"same\") Interaction:      F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}, eta^2 = {r['eta_sq']:.3f}")
r = rt_test_results['A']
print(f"  RT Pair type:              F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}, eta^2 = {r['eta_sq']:.3f}")
r = rt_test_results['B']
print(f"  RT Emotion:                F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}, eta^2 = {r['eta_sq']:.3f}")
r = rt_test_results['AxB']
print(f"  RT Interaction:            F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}, eta^2 = {r['eta_sq']:.3f}")
print()

print('Supplementary SDT:')
for emotion, row in sdt_summary.iterrows():
    print(f"  {emotion:>7}: d'={row['d_prime_M']:.2f} (SD={row['d_prime_SD']:.2f}), "
          f"c={row['criterion_M']:.2f}, hit={row['hit_rate_M']:.2f}, fa={row['fa_rate_M']:.2f}")
print(f"  d' ANOVA: F({df1_d},{df2_d}) = {f_val_d:.3f}, p = {p_d:.3f}, eta^2 = {eta_d:.3f}")
print()

print('Comparison to prior pilots (combined n=12):')
print(f"  Prior d': angry~=-0.05, happy~=-0.08, neutral~=0.01")
print(f"  Current d': angry={sdt_summary.loc['angry','d_prime_M']:.2f}, "
      f"happy={sdt_summary.loc['happy','d_prime_M']:.2f}, "
      f"neutral={sdt_summary.loc['neutral','d_prime_M']:.2f}")

=== Condition 2 Summary (2026-03-21) ===
26 subjects collected (all c2), 1 excluded, 25 analyzed
Design: double study pass (120 trials), balanced 30/30 test split

Study phase (orienting):
  Overall accuracy: 85.9%
  Mean RT: 1057 ms
  Compatibility: F(1,24) = 6.896, p = 0.015
  Emotion:       F(2,48) = 2.670, p = 0.080
  Interaction:   F(2,48) = 0.313, p = 0.733

Test phase (associative recognition):
  p("same") Pair type:       F(1,24) = 1.278, p = 0.270, eta^2 = 0.051
  p("same") Emotion:          F(2,48) = 0.544, p = 0.584, eta^2 = 0.022
  p("same") Interaction:      F(2,48) = 0.455, p = 0.637, eta^2 = 0.019
  RT Pair type:              F(1,24) = 0.987, p = 0.330, eta^2 = 0.039
  RT Emotion:                F(2,48) = 1.694, p = 0.195, eta^2 = 0.066
  RT Interaction:            F(2,48) = 0.115, p = 0.891, eta^2 = 0.005

Supplementary SDT:
    angry: d'=0.14 (SD=0.55), c=-0.07, hit=0.55, fa=0.50
    happy: d'=0.11 (SD=0.66), c=-0.01, hit=0.51, fa=0.48
  neutral: d'=-0.00 (SD=0.45), c=